[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_01_LLM_Fundamentals.ipynb)

# 🧠 Lesson 1: LLM Fundamentals
**Course:** AI/LLM/Agents Engineering — From Java Dev to AI Engineer  
**Date:** 2026-04-29  
**Time to complete:** ~45–60 minutes

---

## What You'll Learn
1. What an LLM actually is (conceptually)
2. How to call the Anthropic API
3. What "tokens" are and why they matter for cost and memory
4. How temperature controls output randomness
5. How to stream responses
6. The most important concept: LLMs are **stateless**

---

## ⚡ Before You Start
You need a free Anthropic API key:
1. Go to **https://console.anthropic.com** and sign up
2. Create an API key (starts with `sk-ant-...`)
3. In Colab, click the **🔑 Secrets** icon (left sidebar)
4. Add a secret named `ANTHROPIC_API_KEY` with your key as the value
5. Toggle **"Notebook access"** ON

Then run the setup cell below. That's the only setup you ever need.

In [ ]:
# ✅ SETUP CELL — Run this first (takes ~10 seconds)
!pip install anthropic -q

import os

# Load your API key from Colab Secrets
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Fallback: paste key directly (not recommended for sharing notebooks)
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-YOUR-KEY-HERE"
    print("⚠️  Using hardcoded key. Use Colab Secrets instead for safety.")

import anthropic
client = anthropic.Anthropic()
print("✅ Anthropic client ready. Let's go!")

---
## Part 1 — What IS an LLM? (The Mental Model)

Think of an LLM as a **very sophisticated next-token predictor**, trained on a massive chunk of human-written text.

When you give it: *"The capital of France is..."*  
It predicts: *"Paris"*

But after training on hundreds of billions of words, this "prediction" becomes incredibly powerful. The model learns grammar, facts, reasoning patterns, code syntax — all as statistical relationships between tokens.

**Key insight for you as a Java engineer:**  
An LLM is NOT a function with a fixed algorithm. It's a **statistical model** with 70–500 billion parameters (learned weights, like millions of tiny dials tuned during training). When you call the API, you're running inference through those weights.

```
Your text → Tokenized → Through neural network layers → Output token probabilities → Sample next token → repeat
```

The output isn't deterministic by default (unlike your Java methods). You'll learn how to control this with `temperature`.

---
### Anatomy of an API Call

| Component | What it is | Java analogy |
|-----------|-----------|------------|
| **model** | Which LLM to use | Which JAR version |
| **system** | Background instructions | Constructor config |
| **messages** | The conversation | Method arguments |
| **max_tokens** | Max response length | Buffer size |
| **temperature** | Randomness 0–1 | Random seed control |

In [ ]:
# ── Exercise 1: Your First API Call ─────────────────────────────────────────

message = client.messages.create(
    model="claude-haiku-4-5-20251001",   # Fast + cheap — perfect for learning
    max_tokens=512,
    messages=[
        {
            "role": "user",
            "content": "What is an LLM in 3 sentences? Explain it to a Java developer."
        }
    ]
)

print("── Response text ───────────────────────────────────")
print(message.content[0].text)

print("\n── Token usage (what you pay for) ──────────────────")
print(f"Input tokens:  {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print(f"Total:         {message.usage.input_tokens + message.usage.output_tokens}")

print("\n── Metadata ────────────────────────────────────────")
print(f"Model:       {message.model}")
print(f"Stop reason: {message.stop_reason}")
# stop_reason == 'end_turn'   → model finished naturally ✅
# stop_reason == 'max_tokens' → response was cut off — increase max_tokens!

In [ ]:
# ── Exercise 1b: Same question, but with a System Prompt ────────────────────
# The system prompt shapes HOW the model responds — tone, format, persona

message2 = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=512,
    system="You are a brutally concise technical teacher. Answer in bullet points only. Max 5 bullets.",
    messages=[
        {"role": "user", "content": "What is an LLM?"}
    ]
)

print(message2.content[0].text)
print(f"\nTokens: {message2.usage.input_tokens} in / {message2.usage.output_tokens} out")

# 💡 EXPERIMENT: Change the system prompt and re-run. Try:
#   "You are a pirate. Explain everything with nautical metaphors."
#   "You are a senior Anthropic engineer. Be very technical."
#   "You are teaching a 10-year-old. Use simple words."

---
## Part 2 — Tokens: The Currency of LLMs

Tokens are **chunks of text** — not exactly words. The model processes everything as tokens.

- `"Hello"` → 1 token  
- `"Hello, world!"` → 4 tokens  
- `"unbelievable"` → 3 tokens (`un`, `believ`, `able`)  
- 1 token ≈ 0.75 words on average

**Why tokens matter:**
- **Cost:** You pay per token (input + output)
- **Context window:** Every model has a max token limit. Claude's is 200,000 tokens (~300 pages). Everything — your prompt, history, and the response — must fit inside it.
- **Memory management:** As conversations grow, so does token count. Managing this is a core agent engineering skill.

In [ ]:
# ── Exercise 2: Token Counting ───────────────────────────────────────────────
# count_tokens() lets you check token count BEFORE spending money on a real call

test_strings = [
    "Hello",
    "Hello, world!",
    "unbelievable",
    "I am a Java developer learning about LLMs and AI agents.",
    "for (int i = 0; i < 10; i++) { System.out.println(i); }",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

print(f"{'Tokens':>6}  {'Words':>5}  {'Ratio':>6}  Text")
print("-" * 70)

for text in test_strings:
    response = client.messages.count_tokens(
        model="claude-haiku-4-5-20251001",
        messages=[{"role": "user", "content": text}]
    )
    tokens = response.input_tokens
    words = len(text.split())
    ratio = tokens / words if words > 0 else 0
    print(f"{tokens:>6}  {words:>5}  {ratio:>6.2f}  '{text[:55]}'")

In [ ]:
# ── Exercise 2b: Cost Calculation ───────────────────────────────────────────
# Haiku pricing (as of early 2026):
#   Input:  $0.25 per 1M tokens
#   Output: $1.25 per 1M tokens

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    messages=[{"role": "user", "content": "Name 5 programming languages, one per line."}]
)

input_tokens  = response.usage.input_tokens
output_tokens = response.usage.output_tokens

input_cost  = input_tokens  * 0.00000025
output_cost = output_tokens * 0.00000125
total_cost  = input_cost + output_cost

print(response.content[0].text)
print(f"\n── Cost breakdown ──────────────────────────────────")
print(f"Input:  {input_tokens} tokens  → ${input_cost:.8f}")
print(f"Output: {output_tokens} tokens  → ${output_cost:.8f}")
print(f"Total:  ${total_cost:.8f}  (way under 1 cent!)")
print(f"\nAt scale: 100,000 calls like this ≈ ${total_cost * 100_000:.2f}")

# 💡 EXPERIMENT: Try the same prompt with claude-sonnet-4-6 instead.
#   Sonnet costs ~5x more but produces better quality.
#   Most production systems use Haiku for simple tasks, Sonnet for complex ones.

---
## Part 3 — Temperature: Controlling Randomness

Temperature is a number **0.0 → 1.0** that controls how "creative" vs. "deterministic" the model is.

- **0.0** → Always picks the highest-probability token. Same input = same output every time.
- **0.5** → Some randomness mixed in.
- **1.0** → Maximum sampling randomness. Creative, varied, but potentially incoherent.

**Java analogy:** `temperature=0` is like `new Random(42)` (seeded, deterministic). `temperature=1` is like `new Random()` (truly random each run).

| Use case | Recommended temperature |
|----------|------------------------|
| Math / code / data extraction | 0.0 |
| Q&A, factual answers | 0.0 – 0.2 |
| Customer support bot | 0.2 – 0.4 |
| Brainstorming | 0.7 – 1.0 |
| Creative writing | 0.8 – 1.0 |

**Default in production agents: 0.0 or 0.2** — you want consistency, not surprise.

In [ ]:
# ── Exercise 3a: Creative task — see how temperature changes output ───────────

def ask(question, temperature):
    r = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=80,
        temperature=temperature,
        messages=[{"role": "user", "content": question}]
    )
    return r.content[0].text.strip()

creative_prompt = "Write a one-sentence story about a robot. Just one sentence."

for temp in [0.0, 0.5, 1.0]:
    print(f"\n🌡️  Temperature {temp}:")
    for i in range(3):
        print(f"  [{i+1}] {ask(creative_prompt, temp)}")

In [ ]:
# ── Exercise 3b: Factual task — high temperature can introduce errors ─────────

factual_prompt = "What is 15 * 23? Answer with just the number, nothing else."

for temp in [0.0, 0.5, 1.0]:
    print(f"\n🌡️  Temperature {temp} (correct answer: 345):")
    for i in range(3):
        print(f"  [{i+1}] {ask(factual_prompt, temp)}")

# 💡 KEY INSIGHT: For factual/code tasks, ALWAYS use temperature=0.
#   High temperature can make the model hallucinate wrong facts or numbers.

---
## Part 4 — Streaming: The "Typing" Effect

Without streaming: you wait 5–10 seconds for a long response, then get it all at once. Bad UX.

With streaming: tokens appear as they're generated — like watching someone type. This is exactly how the ChatGPT and Claude web UIs work.

**Java analogy:** non-streaming is reading the entire HTTP response body into a String; streaming is reading from an `InputStream` line by line.

In production agents, you'll almost always use streaming.

In [ ]:
# ── Exercise 4: Streaming ─────────────────────────────────────────────────────
import time

prompt = "Explain Java garbage collection in 5 sentences."

# Without streaming — measure wait time
print("WITHOUT streaming: waiting...", end="", flush=True)
t0 = time.time()
r = client.messages.create(
    model="claude-haiku-4-5-20251001", max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)
print(f" done in {time.time()-t0:.1f}s")
print(r.content[0].text)

print("\n" + "─"*60)
print("WITH streaming: tokens appear as generated...")
print("─"*60)

t0 = time.time()
with client.messages.stream(
    model="claude-haiku-4-5-20251001", max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)
    final = stream.get_final_message()

print(f"\n\n(Finished in {time.time()-t0:.1f}s | "
      f"{final.usage.input_tokens} in / {final.usage.output_tokens} out tokens)")

---
## Part 5 — The Most Important Concept: LLMs are STATELESS

> **LLMs have NO memory between API calls. You must send the FULL conversation history every single time.**

This is the #1 misconception beginners have. There's no session, no cookie, no server-side memory. Every API call is completely independent.

**Java analogy:** Every API call is like calling a `static` method with no side effects. You pass everything in, you get an answer back. The method has zero internal state between calls.

```
❌ WRONG mental model:
   Call 1: "My name is Gourav"    → "Nice to meet you!"
   Call 2: "What's my name?"      → "I don't know your name"  ← !

✅ RIGHT approach:
   Call 2 sends: [{user: "My name is Gourav"}, {assistant: "Nice to meet you!"}, {user: "What's my name?"}]
              → "Your name is Gourav"
```

In [ ]:
# ── Exercise 5: Multi-turn Conversation ───────────────────────────────────────
# This is the FOUNDATION pattern for all chatbots and agents.

conversation_history = []   # YOU manage this. YOU send it every call.

def chat(user_message, system=None):
    """Send a message and get a response, maintaining full history."""
    conversation_history.append({"role": "user", "content": user_message})

    kwargs = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 200,
        "messages": conversation_history   # ← THE WHOLE HISTORY, every call
    }
    if system:
        kwargs["system"] = system

    response = client.messages.create(**kwargs)
    reply = response.content[0].text

    conversation_history.append({"role": "assistant", "content": reply})

    print(f"   📊 tokens this call: {response.usage.input_tokens} in / "
          f"{response.usage.output_tokens} out | "
          f"history: {len(conversation_history)} messages")
    return reply


system = "You are a concise AI tutor for a Java developer learning LLMs. Keep answers to 2 sentences."

exchanges = [
    "Hi! My name is Gourav and I'm learning about LLMs.",
    "What's my name and what am I learning?",             # Tests memory via history
    "What's the most important thing to understand about tokens?",
    "How does that relate to what I'm learning?",          # Requires full context
]

for user_msg in exchanges:
    print(f"\n👤 You: {user_msg}")
    reply = chat(user_msg, system=system if not conversation_history[1:] else None)
    print(f"🤖 AI:  {reply}")

In [ ]:
# ── Inspect what's actually being sent ────────────────────────────────────────
print("Full conversation history (sent to API on every call):")
print("─" * 60)
for i, msg in enumerate(conversation_history):
    role = msg["role"].upper()
    text = msg["content"][:90] + "..." if len(msg["content"]) > 90 else msg["content"]
    print(f"[{i}] {role}: {text}")

print(f"\nTotal messages: {len(conversation_history)}")
print("⚠️  Notice: input token count grows on EVERY turn as history accumulates.")
print("   This is why agent memory management is a critical engineering problem.")
print("   We'll tackle it in Lesson 5: Agent Memory.")

---
## 🏠 Experiments to Try

Don't just read — modify and run! Here are some ideas:

1. **Change the model** in Exercise 1 to `claude-sonnet-4-6`. Notice quality vs. speed difference.
2. **Temperature extremes** — in Exercise 3a, run with temperature `1.0` five times. Notice the variety.
3. **Break the stateless illusion** — comment out the history in `chat()` so each call starts fresh. Watch the model forget your name.
4. **Count your own prompt** — use `count_tokens()` on a long email or document you wrote. How many tokens is it?
5. **Hit `max_tokens`** — set `max_tokens=20` in Exercise 1 and see how the response gets cut off (`stop_reason='max_tokens'`).

---
## 📝 Key Takeaways

- LLMs are **stateless next-token predictors** — powerful because of scale, not magic.
- **Tokens = money + memory.** Always think token efficiency.
- **Temperature 0** for factual/code tasks. Higher for creativity.
- **Streaming** = better UX. Use it in production.
- **You manage history.** The model knows nothing except what you send.

---
## ➡️ Next Lesson: Prompt Engineering

You'll learn how to write prompts that reliably produce the output format and quality you need — including few-shot examples, chain-of-thought reasoning, and XML structuring. This is the skill that separates amateur AI usage from production-grade work.